## CASH Notebook

## Celestial Chase - LVL 1: The Teal Beacon

You've just woken up from cryo-sleep. No memory. No crew. Just you, a dying sun, and a faint signal pulsing from the sky.

One star glows different from the rest. Not white. Not grey. **Teal.**

Find it. Its position holds part of the key. But position alone is not enough - you must also know how much of its face is lit by the sun.

---

**The encoded signal:** `imtlrohag`

**Your task:**
1. Display the image and find the **cyan pixel** - it is the only pixel where `B == 255` and `G == 255` and `R == 0`
2. Read its `(x, y)` coordinates
3. Use `ephem` to compute **Venus's phase** (`int(venus.phase)`) on `2025/6/21 12:00:00` UTC from Zurich (lat=`47.3769`, lon=`8.5417`)
4. Build the key: `key = x * y + int(venus.phase)`
5. Build the permutation: `random.Random(key).shuffle(list(range(len(encoded))))`
6. Reverse the transposition to decode: `decoded[i] = encoded[perm[i]]`


In [ ]:
import base64, cv2, numpy as np
from IPython.display import display, Image as IPImage

# Starfield image embedded directly in this notebook
img_b64 = "iVBORw0KGgoAAAANSUhEUgAAAZAAAAGQCAIAAAAP3aGbAAATi0lEQVR4Ae3BB3CWZb6H4d9/zohrwwKi4oqKru6Kiisq2NYCSO9JaKGXhE4CCT1A6AkkdBI6IZQUehewF1BxRcVdXUXFFRXBgm3VOZMzs2eYwYFIylfe53vv6zIBkWvTU3tbPF5X3tYhrvfq7IVCCZhQVl0H9F8+Z66AAFm/+8nW9Z9Q+XSMj1uVla0IZQIAR5jgeSOnTJ48YqQA3zMBgCNMADzp3aOf3lr1Wv3WgtyVfWI7ya9MAOAIEwA4wgTA29r27JG3eIkgmZwyaPSoWRMnCYAvmRAIL7558ME7awoIgilz54zoP0CQTECgNW3fbuuatfK9dz45ctt11YTAMQXTzCWLB/foKQBBM2DkiDmTp8gfTADgCBMAOMLkoAPvvVvrllsFwDWjp02dOGy4ysoEFG98xoyxiUPkiNTMjJSERHlG7qaNsS1aCoFjAgBHmABEolFTp0waPkKRxQQAjjABgCNMCIKDhz+oWf0mAQgoEwA4wgQAjjCVw95X9te9r7YAICRMAOAIEwA4wgQAjjABPrassKBbVLTgCBMA/+k7LHn+tDS5xgQAjjBB2r3v5fp17hcAbzPB8/a+sr/ufbUFTyrctTOqQUMhJEwA4AgTADjCVLynX3v1sXvuleOSUsenp4wVEOlO/PJzpQrnK6KZULwvfvzhqgsvEvC7Vm/Z3KFZcyH4TECZ/PPTf//52j8qUvRKTFiUkSl4mynidB80cOms2QqH7c8/1/jhvwlAcJgAaf87h2rfVkNAuM3LWdGvcxcVwwT4w4Y9u1vVqy+4zATAk5YVFnSLihZOYwIAR5gQKrOWLhnUvYcAlJUJABxhAgBHmADAESYAcIQJABxhAgBHmEpvxfp1XVq3EQCElimyxPaJz12QJQCRyOQDfZKTFqSlC4CzNu7d07JuPRPC5PMfvr/6oovlYUe/O1n1kooCPMOE8knPWpAU30cAgs8EAI4whcObHx6+88bqgue9+eHhO2+sLuCUl99+6/7b71CYmIo3a+mSQd17CAC8wQQAjjCheFmrcuM7xgqAN5gAwBEmwAN2vPB8o4ceFvC7TAimo9+drHpJRZXGorVrerVrLwBnMKGsorp1LVy2XABCxQQAjjD5z4fHv7yx8pUKhOYdO2xetVoopRXr13Vp3UbB165Xz7WLFguRwgQAjrAJMzPHDE4Qwidv+7a2jZsIwLmYAMARJgBwhAlhFdWta+Gy5QJwhrihQ7Knz9BpTADgCBMAOMIEIKItyc/rEdNWEcEEAIHz7tFPb616rYLD5I45y5cN6NpNAH4rpkf3/CVL5QMmoKw+PP7ljZWvFErshYNvPFTzLqGsTADgCBM8b9+ht+vUuF2ILPk7tsc0aiyUhgnl89Jbbz5wx51yx8qNGzq1bCX/+firE9dfUUkIuYxFCxN79VYgmMIteUJq2pgUAcC5mADAEabIlTg2JWN8qlB62atXxXXoKMBjTIDvLSss6BYVrVPGZ8wYmzhE4fP+F5/ffNXVwhlMAOAIEwA4wuRLS/LzesS0lYelLZif3KevAJzGBACOMAGAI0yAI6bOmzu8X3/Bx0wogenZWUPj4gWEz7ond7V5ooH8zQREusZtY7bn5QvuM7kscWxKxvhUAfAHEwA4wgQUY8Oe3a3q1RfgGSYAcISp3N755Mht11UTQm7K3Dkj+g8Q4BsmX0pJT0tNShaA0hg7PX380CSFjylwZi5ZPLhHTwFAcJgABMIzB157tNY9QjCZAMARJgBwhMkR//72mz9eepkA+JgJAErgub+/fvTo0XZNmip8TKWXlDo+PWWsACC0TADgCBMQBNOzs4bGxQsIKBOAyHLk66+qXX6FIpEJABxhAlBKPRMGL86cKYScCYD3NIhqs6twnfBbJt+YOm/u8H79BcBZJgAIhM++/+6aiy9RMJlwNluffabpI48KgJeYAMARJgBBkL9je0yjxkJAmVAmedu3tW3cRIgsXQf0Xz5nrgKn+6CBS2fNVik1ionekV8gnMEEAI4wwWX/U1T0v2bynh6DBy2ZOUtAQJkABNNXv/5yxXkVhEAwAaeJ6dE9f8lSAZ5kAsJqwMgRcyZPEVACJgBwhAlA+RUVyUwIMhOAcioq0v8zE4LJBKD8iopkJgSZCYggaQvmJ/fpqwDZ9dKLDR54UPAMEwA4wgQvefWf/7j3z38RgLMxAf5z/Of/VD7/D4JrTADgCBPgiE9PfnttxUsFHzMBgCNMADwpd9PG2BYthdOYAMARJsAFWaty4zvGCv5mAgBHmAA4aPvzzzV++G/yGRMQCDteeL7RQw8rEu079HadGrcLHmAqpcNfHqt+ZRUBZ9MwOmpnQaGA4DABCLldL73Y4IEH5W3zclb069xFXmICEDi7971cv879QnCYitGpb5+V8xcIADzD5G2Fu3ZGNWgoABEhe/WquA4dVVYmAHCECQAcYYI/ZK3Kje8YqxDas39fvdp1FEFSMzNSEhKF8DEBEeTg4Q9qVr9JiFAmAHCECQiEdz45ctt11QQXXFNU9JmZHGQCEEzTs7OGxsULgWACUHonfvm5UoXzhdAynTImbdqE5GEKqBGTJ00ZOUoAEAimSJG7aWNsi5YCELlMKKuWnWI3rswVgFAxAY5Iz1qQFN9H8DHTf2UuXpTQs5cAwMNM/3X85/9UPv8PgjvGzZg+bshQRZxX/vHOfX+5TSilXokJizIyFelMABBQedu3tW3cREFgAgBHmBBxUtLTUpOShXIbPmni1FGjBc8wlcac5csGdO2m8lmSn9cjpq0AoJTss++/u+biSxRCx376scoFFyrkxs2YPm7IUMEbUtLTUpOSBZSGyQeeevWVx++9T0AI9Rg8aMnMWUJAmQDAESYA5zJ5zuyRAwYK4WYCAEeY4D8JKWMyUyfIe4799GOVCy6Us5YW5HePjtHZbNizu1W9+kL5mADAESbASz49+e21FS8VcDYmAHCECQAcYQLC5O2PP7r9+hsElJgJAE55/4vPb77qanmVCQDO0Cgmekd+gTzGBACOMEWKC4uKfjSTB+Rt39a2cRMV7+OvTlx/RSUBKCVTOeze93L9OvcLCLQOcb1XZy8UfGnQ6FGzJk7S2ZiA3zUvZ0W/zl0EeIAp0Pbs31evdh0BQKCZAMARJiDQ1mzd0r5pMwHl80Sb1k+uW6/TmE7plZiwKCNTAOBVJkSE97/4/OarrpZXZa3Kje8YK6B8TABwhvSsBUnxfeQxplDpPSRx4YwMAUBZmQDAESYAcIQJrnn29QOP3F1LgP+YAESWXokJizIyFYlMAOAIEwA4whR881fm9O3UWQBQPiYAcIQJxejcr2/OvPkC4BkmAHCECQAcYQIAR5jCqv+I4XOnTBUAlIDpXNZs3dK+aTMhhA4d+bhGtesF4LdMAOAIEwA4wgQg0EZPmzpx2HD53rCJE6aNHqPAMYVQ6y6d16/IEeCU2cuWDuzWXfAAEwA4woQQWpC7sk9sJwE+03tI4sIZGSo3EwA4wuSIoePHTR87Tt6Tkp6WmpQsAMFnAgBHmBA+jWKid+QXCPC9lRs3dGrZSudiQtD8+9tv/njpZQqEyXNmjxwwUIC/mQDAESbAkybNnjVq4CABpzGV2+v/eu/uP90iAAgyE8rq6Hcnq15SUaGy/fnnGj/8NxXv0JGPa1S7XgiQZYUF3aKiBS8xBV/fYcnzp6UJAMrHBACOMJXejIXZQ3rHCSiTNl27rFu+QkDpmeC+z3/4/uqLLhZQYhNmZo4ZnCDXmPBbnfv1zZk3X/CkfYferlPjdsGvTJBO/PJzpQrnC/CNpQX53aNjFEzTs7OGxsUroEwA4AgTADjCJL389lv3336HgqZZh/ZbVq8RAJSPCQAcYQKktdu2tmvSVIC3mYBTDn95rPqVVQR4lclxqZkZKQmJAuCI5h07bF61WmViAoAAeeHgGw/VvEtBYwLgvt5DEhfOyJBXdenfb8XceSo3EwA4wgT8rpFTJk8eMVIIvrihQ7KnzxCKZwLgPynpaalJyXKNqTQ+OnH8hkqVBQDhYAIAR5gAwBGm4Gjavt3WNWsFl83LWdGvcxcBnmFCabx79NNbq14rAOFgAgBHmADAESYAcIQJABxhAhC59r6yv+59tRUpTABCKG3B/OQ+fYUyMXlG5359c+bNFwAUwwQAjjABcM0LB994qOZd8h8TvGTynNkjBwyUv3Xu1zdn3nwBZzABgCNMAOAIk1NWb9ncoVlz+cmK9eu6tG4jAJIJKIcWsR035a4SfKNw186oBg0VJiac4ZkDrz1a6x4hOHI3bYxt0VIoqxkLs4f0jpMvmXCGZh3ab1m9RgDKavPTTzV/7HEFmglAWDVp13bb2jyhBEyl1yCqza7CdQLKKnlCatqYFAGlZAIAR5gAzzivqOhXMwHFMAGAI0wA4AgTgLOJ6dE9f8lSwUtMABBQwyZOmDZ6jILABACOMAFAKT316iuP33ufQs4E1xx4791at9yqwNmwZ3erevUFhMpVRUVfmKn0TO47ePiDmtVvEoBIZwIAR5gAOO7QkY9rVLtePmACfOz4z/+pfP4fBEeYAESu9z47ess1VRUpTADKatr8ecP69hNCxQQgsnz2/XfXXHyJIpEJYTUmbdqE5GGC//QfMXzulKlCaZgAwBEmlNL4jBljE4cIQMiZAMARJgBwhMkFlYqKTpgJgL+ZzmXU1CmTho+Q+8ZOTx8/NElARFict7Zn23YKuWETJ0wbPUZhYgqJXokJizIyBQDlYAIAR5gAwBEmAHCECeeSlDo+PWWs3PHmh4fvvLG6gIhjAsrtg2Nf3FTlKkF68uWXnrj/ASE4TJGiaft2W9eslT+MnjZ14rDhQri9/q/37v7TLUKomCJUw+ionQWFAiLFMwdee7TWPfI3EwA4wgQAjjC5rFXnThtyVgrBtOmpvS0eryvAA0wotxXr11WoUKF902YCEEwmAA7qPSRx4YwMhU/m4kUJPXsptEzwhlWbN3Vs3kIAimcCAEeYAPjV5DmzRw4YKHeYgDJZWpDfPTpGQAiZAA/rNnDAstlzFCpHvv6q2uVXCF5l8quG0VE7CwoFBM6EmZljBico3PYdertOjdsViUylN3fF8v5dugoAQssEAI4wAYAjTPC32D7xuQuyhIDasGd3q3r15VVvfPD+XTfdLAeZAmTP/n31atcREHJ527e1bdxE8AETADjChOBLHJuSMT5VIdR90MCls2YLiCwmAHCECQAcYSqlL3784aoLLxLKZOCokbMnTRaAMjEBKLEtzzzd7NHHFCYfnTh+Q6XK8jGTs5q2b7d1zVoB8A0TEG4fnTh+Q6XKAs7FhBI7+t3JqpdUFIAwMQEh8c4nR267rpqAcjABgCNMiFCNYqJ35BcIiCAmAHCECSiB0dOmThw2XEBYmQDAESavGjxm9MwJEwUAp5gAwBEmAHCECQAcYQIAr2ravt3WNWt1igkAHGECIkJ0924FS5cJJbZ6y+YOzZrLKSYAcIQJEaFlp9iNK3MFRDST533yzdfXXXa5itF1QP/lc+YKgA+YAJddXFT0vZngDyYAcIQJQDEGjR41a+IkwTNMCKutzz7T9JFHBaAETADgCBPgJ2999OEdN9wouMkE17Tu0nn9ihwB/mM6Td9hyfOnpQkAPMkEOOvTk99eW/FSBd+B996tdcutQriZ4D8No6N2FhQKcI3JH7oNHLBs9hwBcJkJCJwegwctmTlLgbB229Z2TZoKOI0JAILp8JfHql9ZRYFgAty3OG9tz7bthHI78cvPlSqcL68yAYgIn3zz9XWXXa6IZsJ/JaWOT08ZKwAeZgqHwl07oxo0FIAzLFyzunf7DvKBZ18/8MjdtVQaJvjJuBnTxw0ZKkB69vUDj9xdS04xAYAjTCXw1a+/XHFeBXnYsZ9+rHLBhXJfUur49JSxAkJi01N7WzxeV+4wAYAjTHDNivXrurRuI8B/TECE2vLM080efUyIICYAcIQpyNZs3dK+aTMh+OKThmalTxectfeV/XXvqy0UzwQAjjAVb9NTe1s8XlcA4A0mAIEwYWbmmMEJQjCZAMARJgTf82/8/eG7/irgXJ5o0/rJdeuFYpgAwBEmAPCkQ0c+rlHtep3GBKDc9uzfV692HSHITAAQJnv276tXu45KzAQAAZWSnpaalKwgMKFkGreN2Z6XLwDhYwJwhopFRSfNVAId4+NWZWXLfTMWZg/pHSdvMwEo3uK8tT3bthO8wQTAeybOmjl60GDht0wA4AgTADjCBACOMAGAI0wIhD7JSQvS0vVb+Tu2xzRqLAABYgIQCPNX5vTt1FkIJhOAMpm5ZPHgHj2FEDIBLsjfsT2mUWPB30wA4AgTgFL66tdfrjivghByJuBscjas79yqtQAvMSE4Bo4aOXvSZAXHiMmTpowcJYTWvJwV/Tp3EcLHBACOMAH+kLtpY2yLloLLTADgCBMAOMIEAI4wAYAjTADgJS8cfOOhmnfpbEwAivfqP/9x75//Ip9ZvWVzh2bNFT7NOrTfsnqNzmACAEeYAPheVLeuhcuWy/NMAOAIUxCs3LihU8tWAoCAMgFAMO166cUGDzyoQDABgCNMAOAIEwA/mThr5uhBg+UmU7l1jI9blZUtnMvufS/Xr3O/AJSVCQDOpUFUm12F6xRuJgBwhEk69tOPVS64UADgbSb4WIvYjptyVwlwxP8Bz0axzWOG3BAAAAAASUVORK5CYII="

img_bytes = base64.b64decode(img_b64)
img_arr   = np.frombuffer(img_bytes, dtype=np.uint8)
img       = cv2.imdecode(img_arr, cv2.IMREAD_COLOR)

cv2.imwrite('starfield.png', img)
display(IPImage('starfield.png'))
print("Image shape:", img.shape)


In [ ]:
import ephem, random

encoded  = "imtlrohag"
obs_date = "2025/6/21 12:00:00"
obs_lat  = "47.3769"
obs_lon  = "8.5417"

# TODO Step 1: Find the cyan pixel (B=255, G=255, R=0) in img
# Hint: use np.where or a loop
pixel_x, pixel_y = 0, 0  # replace with actual coordinates

# TODO Step 2: Compute Venus phase with ephem
obs = ephem.Observer()
obs.lat  = obs_lat
obs.lon  = obs_lon
obs.date = obs_date
venus = ephem.Venus()
venus.compute(obs)
phase = 0  # replace: int(venus.phase)

# TODO Step 3: Build key and permutation
key  = pixel_x * pixel_y + phase
perm = list(range(len(encoded)))
random.Random(key).shuffle(perm)

# TODO Step 4: Reverse the transposition
# Hint: if encoded[perm[i]] = original[i], then decoded[i] = encoded[perm[i]]? Think carefully.
def transpose_decode(encoded, perm):
    pass  # implement this

answer = transpose_decode(encoded, perm)
print(answer)
